<a href="https://colab.research.google.com/github/renatofb98/Data_science_projects/blob/main/Ler_Manutencoes_para_Excel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Leitor de relatório de Manutenções (GCASPP) → Excel

Este notebook lê o relatório em PDF de manutenções de veículos (formato
"PREFEITURA MUNICIPAL DE ... / ALMOXARIFADO / GARAGEM MUNICIPAL", sistema
GCASPP) e gera uma planilha Excel estruturada.

**Por que OCR e não extração de texto direta?**
Esse tipo de relatório, quando exportado do sistema pelo navegador
("Salvar como PDF"), às vezes sai com o texto convertido em curvas/vetores
em vez de texto real — nesse caso `pdftotext`/`PyPDF2`/`pdfplumber` não
extraem nada além do cabeçalho da página. O notebook detecta isso e, se
necessário, usa OCR (Tesseract). Se o seu PDF tiver texto real, ele
tentará a extração direta primeiro (mais rápida e sem erro de OCR).

**Passos:** rode as células em ordem. Na célula 3 você vai enviar o PDF.


In [1]:
# 1) Dependências de sistema (Colab: roda uma vez por sessão)
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-por
!pip -q install pdfplumber openpyxl pandas


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.9_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.9) ...
Selecting previously unselected package tesseract-ocr-por.
Preparing to unpack .../tesseract-ocr-por_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-por (1:4.1.0-2) ...
Setting up tesseract-ocr-por (1:4.1.0-2) ...
Setting up poppler-utils (24.02.0-1ubuntu9.9) ...
Processing triggers for man-db (2.12.0-4build2) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 714.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━

In [2]:
# 2) Envie o PDF do relatório de manutenções
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Arquivo recebido:", PDF_PATH)


Saving Manutenções 01092025 a 01092026.pdf to Manutenções 01092025 a 01092026 (1).pdf
Arquivo recebido: Manutenções 01092025 a 01092026 (1).pdf


In [3]:
# 3) Núcleo do parser
import re
import subprocess
import tempfile, os, glob
import pandas as pd

DPI = 300
LANG = "por"

VEH_RE = re.compile(
    r'^(?P<code>\d+)\s*-\s*(?P<platemodel>.+?)\s+'
    r'(?P<year>(?:(?:19|20)\d{2}|\d{2}))\s+'
    r'(?P<dept>[A-ZÀ-Ú][A-ZÀ-Ú /.\-]*)$'
)
TOTAL_RE = re.compile(r'^Total:\s*(?P<count>\d+)\b')
DATA_RE = re.compile(
    r'^(?P<data>\d{2}/\d{2}/\d{4})\s+(?P<prestadora>.+?)\s+(?P<qtd>\d+)\s+'
    r'(?:0,00\s+){0,7}(?P<vloutros>[\d.]+,\d{2})\s+(?P<totalocorr>[\d.]+,\d{2})\s*$'
)
BOILER_RE = re.compile(
    r'^(PREFEITURA|ALMOXARIFADO|GARAGEM MUNICIPAL|GCASPP|Descrição Ocorrência|'
    r'Data\s+KM|Exercício|Página|.+,\s*(Segunda|Terça|Quarta|Quinta|Sexta|Sábado|Domingo)-feira,)'
)
OCC_LABEL_RE = re.compile(r'^[A-ZÀ-Ú][A-ZÀ-Ú /.]*$')


def to_float(s: str) -> float:
    return float(s.replace('.', '').replace(',', '.'))


def has_real_text_layer(pdf_path: str) -> bool:
    """Testa se o PDF tem texto extraível de verdade (além do cabeçalho)."""
    try:
        out = subprocess.run(
            ["pdftotext", "-f", "2", "-l", "2", "-layout", pdf_path, "-"],
            capture_output=True, text=True, check=True,
        ).stdout
    except subprocess.CalledProcessError:
        return False
    # se sobrar pouco texto além do cabeçalho repetido, provavelmente é vetor/curva
    return len(out.strip()) > 300


def ocr_pdf(pdf_path: str, dpi: int = DPI, lang: str = LANG) -> str:
    with tempfile.TemporaryDirectory() as tmp:
        prefix = os.path.join(tmp, "pg")
        subprocess.run(["pdftoppm", "-r", str(dpi), "-png", pdf_path, prefix], check=True)
        pages = sorted(glob.glob(prefix + "*.png"))
        chunks = []
        for p in pages:
            res = subprocess.run(
                ["tesseract", p, "stdout", "-l", lang,
                 "--psm", "6", "-c", "preserve_interword_spaces=1"],
                capture_output=True, text=True,
            )
            chunks.append(res.stdout)
        return "\n".join(chunks)


def extract_text_direct(pdf_path: str) -> str:
    res = subprocess.run(["pdftotext", "-layout", pdf_path, "-"],
                          capture_output=True, text=True, check=True)
    return res.stdout


def parse(text: str) -> pd.DataFrame:
    rows = []
    veh = {"code": None, "plate": None, "model": None, "dept": None}
    occ_type = None

    for raw in text.splitlines():
        l = raw.strip()
        if not l or BOILER_RE.match(l) or l.replace(" ", "") == "ALMOXARIFADO":
            continue
        if TOTAL_RE.match(l):
            continue  # subtotais recalculados no Excel

        m = VEH_RE.match(l)
        if m:
            platemodel = m.group("platemodel").strip()
            parts = platemodel.split(None, 1)
            plate = parts[0] if parts else ""
            model = parts[1] if len(parts) > 1 else ""
            veh = {
                "code": m.group("code"),
                "plate": plate,
                "model": f"{model} {m.group('year')}".strip(),
                "dept": m.group("dept").strip(),
            }
            occ_type = None
            continue

        m = DATA_RE.match(l)
        if m:
            rows.append({
                "Cód. Veículo": veh["code"],
                "Placa": veh["plate"],
                "Modelo/Ano": veh["model"],
                "Secretaria/Setor": veh["dept"],
                "Tipo Ocorrência": occ_type,
                "Data": pd.to_datetime(m.group("data"), format="%d/%m/%Y", errors="coerce"),
                "Prestadora Serviço": m.group("prestadora").strip(),
                "Qtd. Item": int(m.group("qtd")),
                "Vl Outros (R$)": to_float(m.group("vloutros")),
                "Total Ocorrência (R$)": to_float(m.group("totalocorr")),
            })
            continue

        if OCC_LABEL_RE.match(l):
            occ_type = l
            continue

    return pd.DataFrame(rows)


print("Verificando se o PDF tem texto real ou precisa de OCR...")
if has_real_text_layer(PDF_PATH):
    print("→ Texto real encontrado, extraindo direto (rápido).")
    texto = extract_text_direct(PDF_PATH)
else:
    print("→ Sem texto extraível (provável texto vetorizado) — rodando OCR. "
          "Isso pode levar alguns minutos em PDFs longos.")
    texto = ocr_pdf(PDF_PATH)

df = parse(texto)
print(f"\nLinhas de manutenção extraídas: {len(df)}")
print(f"Soma de 'Total Ocorrência': R$ {df['Total Ocorrência (R$)'].sum():,.2f}")
df.head(10)


Verificando se o PDF tem texto real ou precisa de OCR...
→ Sem texto extraível (provável texto vetorizado) — rodando OCR. Isso pode levar alguns minutos em PDFs longos.

Linhas de manutenção extraídas: 844
Soma de 'Total Ocorrência': R$ 380,777.52


,Cód. Veículo,Placa,Modelo/Ano,Secretaria/Setor,Tipo Ocorrência,Data,Prestadora Serviço,Qtd. Item,Vl Outros (R$),Total Ocorrência (R$)
0,3,CDV-1467,GM/ S10O 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO PNEU,2025-10-16,BORRACHARIA E PNEUS FORMIGA,0,24.0,24.0
1,3,CDV-1467,GM/ S10O 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-05-25,MAICO FERNANDO BOLOGNESI,0,1130.0,1130.0
2,3,CDV-1467,GM/ S10O 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-13,JOAO BATISTA BUENO PINTO,0,750.0,750.0
3,3,CDV-1467,GM/ S10O 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-22,BORRACHARIA E PNEUS FORMIGA,0,671.0,671.0
4,3,CDV-1467,GM/ S10O 2.4 RONTAN CAMIONETA 2002,OBRAS,REPARO,2026-07-27,AUTO PECAS E MECANICA J.R.R.,0,294.0,294.0
5,6,DBASC21,FIAT/ FIORINO IE 2005,OBRAS,REPARO,2025-10-28,RIMENIS BALDACIN OLIVEIRA,0,160.0,160.0
6,6,DBASC21,FIAT/ FIORINO IE 2005,OBRAS,REPARO PNEU,2025-12-10,BORRACHARIA E PNEUS FORMIGA,0,149.0,149.0
7,6,DBASC21,FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-04-29,BORRACHARIA E PNEUS FORMIGA,0,27.0,27.0
8,6,DBASC21,FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-05-25,UNITEC ABC MANUTENCAO VEIC,0,260.0,260.0
9,6,DBASC21,FIAT/ FIORINO IE 2005,OBRAS,REPARO,2026-06-16,SCUDELER PNEUS E CENTRO,0,570.0,570.0


**Confira o total acima contra o rodapé do PDF original (linha "Total: N ...").**
Se baterem, a extração está correta. Se o PDF tiver colunas diferentes
(ex.: nomes de campo distintos), ajuste as regex `VEH_RE` / `DATA_RE` na
célula anterior — elas foram calibradas para o layout
`Data | KM Ocor. | Próx. KM | Prestadora Serviço | Qtd. Item | Vl. Unitário |
Vl. Óleo | Filt. Óleo | Filt Comb A | Filt Comb B | Filtro Ar | Óleo Dif. |
Vl Outros | Total Ocorr.`


In [4]:
# 4) Monta o Excel (dados + resumos)
OUT_XLSX = "Manutencoes.xlsx"

resumo_veiculo = (
    df.groupby(["Cód. Veículo", "Placa", "Modelo/Ano", "Secretaria/Setor"], dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Gasto=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Gasto", ascending=False)
)
resumo_prestadora = (
    df.groupby("Prestadora Serviço", dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Recebido=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Recebido", ascending=False)
)
resumo_setor = (
    df.groupby("Secretaria/Setor", dropna=False)
    .agg(Qtd_Ocorrencias=("Total Ocorrência (R$)", "count"),
         Total_Gasto=("Total Ocorrência (R$)", "sum"))
    .reset_index().sort_values("Total_Gasto", ascending=False)
)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Dados", index=False)
    resumo_veiculo.to_excel(writer, sheet_name="Resumo por Veículo", index=False)
    resumo_prestadora.to_excel(writer, sheet_name="Resumo por Prestadora", index=False)
    resumo_setor.to_excel(writer, sheet_name="Resumo por Setor", index=False)

from openpyxl import load_workbook
wb = load_workbook(OUT_XLSX)
for ws in wb.worksheets:
    for col in ws.columns:
        length = max((len(str(c.value)) for c in col if c.value is not None), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(length + 2, 45)
wb.save(OUT_XLSX)

print("Excel gerado:", OUT_XLSX)


Excel gerado: Manutencoes.xlsx


In [5]:
# 5) Baixa o Excel
from google.colab import files
files.download(OUT_XLSX)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6) Adicionar o Excel ao GitHub

Para adicionar o arquivo Excel ao seu repositório GitHub, siga estes passos:

1.  Instale o `git`.
2.  Configure seu nome de usuário e e-mail do `git`.
3.  Clone seu repositório GitHub. Você precisará substituir `YOUR_GITHUB_REPO_URL` pelo URL real do seu repositório. Para autenticação, você pode usar um [Personal Access Token (PAT)](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token) no formato `https://<YOUR_PERSONAL_ACCESS_TOKEN>@github.com/username/repo.git` ou configurar SSH.
4.  Copie o arquivo Excel gerado para o diretório do repositório clonado.
5.  Adicione, `commit` e `push` as alterações.

In [6]:
# Instala o git
!apt-get update -qq
!apt-get install git -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
# Configura o nome de usuário e e-mail do git
# Substitua pelo seu nome e e-mail do GitHub
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

### Clone o seu repositório GitHub

**ATENÇÃO**: Substitua `YOUR_GITHUB_REPO_URL` pelo URL HTTPS do seu repositório. Se for um repositório privado ou se você tiver autenticação de dois fatores habilitada, você precisará usar um Personal Access Token (PAT) no lugar da sua senha, ou incluí-lo diretamente no URL, por exemplo: `https://<YOUR_PERSONAL_ACCESS_TOKEN>@github.com/username/repo.git`. Certifique-se de que o repositório esteja vazio ou que você esteja clonando em um diretório seguro.

In [ ]:
REPO_DIR = "my_github_repo"  # Nome do diretório onde o repositório será clonado
GITHUB_REPO_URL = "YOUR_GITHUB_REPO_URL" # <-- SUBSTITUA COM O SEU URL DO REPOSITÓRIO GITHUB

# Remove o diretório do repositório se ele já existir (para evitar erros em execuções repetidas)
!rm -rf {REPO_DIR}

# Clona o repositório
!git clone {GITHUB_REPO_URL} {REPO_DIR}

# Entra no diretório do repositório
%cd {REPO_DIR}

# Copia o arquivo Excel gerado para o diretório do repositório
import shutil
shutil.copy(f"/content/{OUT_XLSX}", OUT_XLSX)

# Adiciona o arquivo, faz o commit e empurra para o GitHub
!git add {OUT_XLSX}
!git commit -m "Add Manutencoes.xlsx report"
!git push origin main # ou 'master', dependendo do nome da sua branch principal

# Volta para o diretório raiz do Colab
%cd /content